# Anomaly detection


Xiao & Fan (2025) found that what counts the most for anomaly detection was embedding quality, and that deep-learning based approaches showed little to no improvement over "shallow" algorithms like KNN and Isolation Forest.

This section starts with using said shallow algorims.

I will be using many codes and coding tutorials as reference for this section.

The main reference is Xiao and Fan's code on github: https://github.com/jicongfan/Text-Anomaly-Detection-Benchmark/blob/main/anomaly_detection/main.py

However, since their code provides little explanation, and since I'm learning as I go. I am more comfortable at my current level to also follow tutorials which explain in detail each step. The additional tutorials will be linked accordingly.

# Prepping

Since it's a new day I have to re-load everything again and prep the embeddings.

In [ ]:
#Going to import everything again

import torch
import pandas as pd
import numpy as np

from transformers import AutoModel, DistilBertTokenizer


In [ ]:
#loading model that was pushed to Hugging Face 🤗

mrr_model = "limhayi/mrr-punk-columns-and-letters-distilbert"

tokenizer = DistilBertTokenizer.from_pretrained(mrr_model)

model = AutoModel.from_pretrained(mrr_model) #remove masked MLM here because now I'm not fine-tuning, different task.

In [ ]:
print(type(tokenizer)) #double-checking tokenizer

In [ ]:
%pip install datasets --upgrade

There is a bug with Datasets that requires me to have the latest version. Check your version when running this.

In [ ]:
from datasets import Dataset

In [ ]:
df = pd.read_excel("/content/finalized_joined_columns.xlsx")

In [ ]:
df.info()

In [ ]:
hf_dataset = Dataset.from_pandas(df)

First we tokenize everything again to extract the hidden states.

In [ ]:
def tokenize_for_embeddings(batch):
  return tokenizer(batch["Text"],
                   padding = "max_length",
                   truncation = True,
                   max_length = 512,
                   return_overflowing_tokens = True,
                   stride = 128
  )



In [ ]:
tokenized_dataset = hf_dataset.map(
    tokenize_for_embeddings,
    batched = True,
    remove_columns = [col for col in hf_dataset.column_names if col != 'overflow_to_sample_mapping']
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Now feature extraction.

In [ ]:
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "overflow_to_sample_mapping"])

Embeddings should be created now using mean pooling.

The [CLS] approach or max pooling is designed for classification tasks, but it may not capture sentence meaning.

I am taking the mean pooling function from here: https://discuss.huggingface.co/t/get-word-embeddings-from-transformer-model/6929



In [ ]:
def extract_hidden_states(batch):
    inputs = {k: v.to(device) for k, v in batch.items()
              if k in tokenizer.model_input_names}
    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = model(**inputs).last_hidden_state
        attention_mask = inputs['attention_mask']

        #mean pooling

    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

    sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)


    sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
    mean_pooled = sum_embeddings / sum_mask


    return {"hidden_state": mean_pooled.cpu().numpy()}



In [ ]:
hidden_states_dataset = tokenized_dataset.map(extract_hidden_states,
                                    batched=True,
                                    batch_size= 16)

Re-running the code and ran into a bug, the same bug has been reported in March 2026: https://github.com/huggingface/datasets/issues/8085

Please make sure that you are running the latest version of datasets (last checked 18/06/2026)

Sorry for the incovenience :(

In [ ]:
print(hidden_states_dataset.column_names)

Turn everything now into a numpy matrix so that it can run with anomaly detection.

In [ ]:
X_full = np.array(hidden_states_dataset["hidden_state"])


print(f"Matrix ready! Shape: {X_full.shape}")

yay it's a normal shape, so nothing went wrong. Things usually go wrong by now.

In [ ]:
print("Sample of X_full")
print(X_full[:5, :10])

Some preliminary t-SNE visualization to check how things are looking before Isolation Forest.

https://distill.pub/2016/misread-tsne/ -> provides a good guide on how to read and what parameters to set for t-SNE

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, perplexity=50, random_state=42, learning_rate = 'auto')
X_embedded = tsne.fit_transform(X_full)

plt.figure(figsize=(10, 8))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], alpha=0.5, s=10, color = 'blue')
plt.title("t-SNE projection")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

Same data but with slightly different parameters to experiment with t-SNE

In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=22, learning_rate = 'auto')
X_embedded = tsne.fit_transform(X_full)

plt.figure(figsize=(10, 8))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], alpha=0.5, s=10, color = 'hotpink')
plt.title("t-SNE projection")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

# Isolation Forest

*Isolation Forest. It is a tree based 🌲 algorithm. It works on the notion that anomalies are easier to isolate.

It is unsupervised and works with high-dimensional data, which is perfect for my use-case.

Tutorial I followed: https://medium.com/@yanivbohbot5/anomaly-detection-with-isolation-forest-a-complete-guide-d42da77510a6

But the sklearn documentation already offers enough information: https://scikit-learn.org/stable/auto_examples/ensemble/plot_isolation_forest.html

In [ ]:
#X_full already ran

from sklearn.ensemble import IsolationForest

In [ ]:
#Fit the model

iso_forest = IsolationForest(n_estimators = 100,
                      contamination = 0.02, #play around, but this determines the % of items in the dataset that will get flagged
                      random_state = 55)

The contamination parameter determines the percentage of anomalies to be detected in the dataset.

In [ ]:
iso_predictions = iso_forest.fit_predict(X_full)

In [ ]:
comparison_df = pd.DataFrame(index=range(len(hidden_states_dataset)))

In [ ]:
comparison_df['IsoForest_Raw'] = iso_predictions

comparison_df['IsoForest'] = comparison_df['IsoForest_Raw'] == -1

Gemini helped me with producing the following interactive visualization:

In [ ]:
import plotly.express as px
import pandas as pd
from sklearn.manifold import TSNE

# Re-calculate t-SNE for the letters data (X_full_2) if it hasn't been run recently
tsne = TSNE(
    n_components=2,
    perplexity=50,
    early_exaggeration=15,
    learning_rate=200,
    random_state=42,
    init='pca'
)
X_embedded = tsne.fit_transform(X_full)

# Optional: Mathematical Stretching for better visualization spread
X_embedded[:, 0] *= 1.5
X_embedded[:, 1] *= 1.5

# Prepare the plot data from the comparison_df (which is currently for letters)
plot_df = comparison_df.copy()
plot_df['tsne_1'] = X_embedded[:, 0]
plot_df['tsne_2'] = X_embedded[:, 1]

# Get chunk texts for hover info
plot_df['Chunk_Text'] = [tokenizer.decode(ids, skip_special_tokens=True) for ids in hidden_states_dataset['input_ids']]
plot_df['Snippet'] = plot_df['Chunk_Text'].str.slice(0, 500) + "..."
plot_df['Chunk_ID'] = plot_df.index

# Categorize points based ONLY on IsoForest predictions
plot_df['Anomaly_Status'] = plot_df['IsoForest'].apply(lambda x: 'IsoForest Anomaly' if x else 'Normal')

# Create the interactive plot
fig = px.scatter(
    plot_df,
    x='tsne_1',
    y='tsne_2',
    color='Anomaly_Status',
    color_discrete_map={
        'Normal': '#B0BEC5',            # Light Gray for normal points
        'IsoForest Anomaly': '#E34234'  # Vermillion
    },
    custom_data=['Snippet', 'Chunk_ID'],
    title='<b>Isolation Forest Anomalies in Letters Dataset</b>',
    template='plotly_white'
)

# Refine markers and hover tooltip
fig.update_traces(
    marker=dict(
        line=dict(width=0.8, color='white'),
        opacity=0.85
    ),
    hovertemplate="<b>Chunk ID: %{customdata[1]}</b><br>Status: %{fullData.name}<br><br><i>%{customdata[0]}</i><extra></extra>"
)

fig.update_layout(
    width=1200,
    height=800,
    xaxis=dict(showgrid=True, zeroline=True, showticklabels=True, title='t-SNE Dimension 1'),
    yaxis=dict(showgrid=True, zeroline=True, showticklabels=True, title='t-SNE Dimension 2'),
    legend_title="Anomaly Type"
)

fig.write_html("isoforest_anomalies_letters_map.html")
fig.show()


To make my life easier I want to extract the anomalies and place them in a dataframe alongside their coordinates.

In [ ]:
anomalies_df = plot_df[plot_df['IsoForest'] == True].copy()

In [ ]:
anomalies_df = anomalies_df[['tsne_1', 'tsne_2', 'Chunk_ID', 'Snippet']]

In [ ]:
anomalies_df.info()

In [ ]:
anomalies_df.to_excel("opinion_columns_anomalies.xlsx")

# Now the letters


It is just a repetition of the same steps taken for the opinion columns

In [ ]:
import pandas as pd
df_letterz = pd.read_excel("/content/finalized_mrr_letters.xlsx")

In [ ]:
df_letterz = df_letterz[df_letterz['letter_content'].str.split().str.len() > 80].copy() #solving the short letters issue

When I initially ran this IsoForest marked most anomalies simply because they were short letters, so I'm adding a filter.

In [ ]:
df_letterz.info()

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(df_letterz[['letter_content']])

In [ ]:
def tokenize_for_embeddings(batch):
  return tokenizer(batch["letter_content"],
                   padding = "max_length",
                   truncation = True,
                   max_length = 512,
                   return_overflowing_tokens = True,
                   stride = 128
  )




In [ ]:
tokenized_dataset = hf_dataset.map(
    tokenize_for_embeddings,
    batched = True,
    remove_columns = [col for col in hf_dataset.column_names if col != 'overflow_to_sample_mapping']
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "overflow_to_sample_mapping"])

In [ ]:
def extract_hidden_states(batch):
    inputs = {k: v.to(device) for k, v in batch.items()
              if k in tokenizer.model_input_names}
    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = model(**inputs).last_hidden_state
        attention_mask = inputs['attention_mask']

        #mean pooling

    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

    sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)


    sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
    mean_pooled = sum_embeddings / sum_mask


    return {"hidden_state": mean_pooled.cpu().numpy()}


In [ ]:
hidden_states_dataset = tokenized_dataset.map(extract_hidden_states,
                                    batched=True,
                                    batch_size= 16)

In [ ]:
print(hidden_states_dataset.column_names)

In [ ]:
X_full_2 = np.array(hidden_states_dataset["hidden_state"])


print(f"Matrix ready! Shape: {X_full_2.shape}")

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, perplexity=25, random_state=77, learning_rate='auto')
X_embedded = tsne.fit_transform(X_full_2)

plt.figure(figsize=(10, 8))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], alpha=0.5, s=10)
plt.title("t-SNE projection")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

# Anomaly detection for letters

In [ ]:
from sklearn.ensemble import IsolationForest

In [ ]:
iso_forest = IsolationForest(n_estimators = 100,
                      contamination = 0.02, #play around
                      random_state = 55)

In [ ]:
iso_predictions = iso_forest.fit_predict(X_full_2)

In [ ]:
comparison_df = pd.DataFrame(index=range(len(hidden_states_dataset)))

In [ ]:
comparison_df['IsoForest_Raw'] = iso_predictions

In [ ]:
comparison_df['IsoForest'] = comparison_df['IsoForest_Raw'] == -1

These are the settings I used in the end for the visualizations, makes the anomalies slighly larger.

In [ ]:
import plotly.express as px
import pandas as pd
from sklearn.manifold import TSNE

# Re-calculate t-SNE for the letters data (X_full_2) if it hasn't been run recently
tsne = TSNE(
    n_components=2,
    perplexity=30,
    early_exaggeration=15,
    learning_rate='auto',
    random_state=22,
    init='pca'
)
X_embedded = tsne.fit_transform(X_full_2)

# Optional: Mathematical Stretching for better visualization spread
X_embedded[:, 0] *= 1.5
X_embedded[:, 1] *= 1.5

# Prepare the plot data from the comparison_df (which is currently for letters)
plot_df = comparison_df.copy()
plot_df['tsne_1'] = X_embedded[:, 0]
plot_df['tsne_2'] = X_embedded[:, 1]

# Get chunk texts for hover info
plot_df['Chunk_Text'] = [tokenizer.decode(ids, skip_special_tokens=True) for ids in hidden_states_dataset['input_ids']]
plot_df['Snippet'] = plot_df['Chunk_Text'].str.slice(0, 500) + "..."
plot_df['Chunk_ID'] = plot_df.index

# Categorize points based ONLY on IsoForest predictions
plot_df['Anomaly_Status'] = plot_df['IsoForest'].apply(lambda x: 'IsoForest Anomaly' if x else 'Normal')

# Create the interactive plot
fig = px.scatter(
    plot_df,
    x='tsne_1',
    y='tsne_2',
    color='Anomaly_Status',
    color_discrete_map={
        'Normal': '#B0BEC5',            #  Gray
        'IsoForest Anomaly': '#E34234' #vermillion #'#FF7F50'  # Neon Red for IsoForest anomalies
    },
    size=plot_df['Anomaly_Status'].apply(lambda x: 2 if x == 'Normal' else 3.3), # Make anomalies larger
    size_max=10,
    custom_data=['Snippet', 'Chunk_ID'],
    title='<b>Isolation Forest Anomalies in Letters Dataset</b>',
    template='plotly_white'
)


# Refine markers and hover tooltip
fig.update_traces(
    marker=dict(
        line=dict(width=0.8, color='white'),
        opacity=0.85
    ),
    hovertemplate="<b>Chunk ID: %{customdata[1]}</b><br>Status: %{fullData.name}<br><br><i>%{customdata[0]}</i><extra></extra>"
)

fig.update_layout(
    width=1200,
    height=800,
    xaxis=dict(showgrid=True, zeroline=True, showticklabels=True, title='t-SNE Dimension 1'),
    yaxis=dict(showgrid=True, zeroline=True, showticklabels=True, title='t-SNE Dimension 2'),
    legend_title="Anomaly Type"
)

fig.write_html("isoforest_anomalies_letters_map.html")
fig.show()


In [ ]:
anomalies_df = plot_df[plot_df['IsoForest'] == True].copy()

In [ ]:
anomalies_df = anomalies_df[['tsne_1', 'tsne_2', 'Chunk_ID', 'Snippet']]

In [ ]:
anomalies_df.info()

In [ ]:
anomalies_df.to_excel('letter_anomalies.xlsx')